# 01 Calibration and Validation

SCE-UA calibration of the reduced critical-zone structure model against
first-arrival travel times on the seven Treeline seismic profiles, and
prediction on the two held-out profiles.

Lines are labelled `TL1` to `TL7`. `TL1` to `TL5` enter the calibration
objective; `TL6` and `TL7` are held out and predicted after calibration.

## What This Notebook Produces

- the best-fitting parameter set and the behavioral ensemble retained from the
  best 5% of evaluations
- per-line travel-time RMSE and normalized RMSE for the calibration and
  held-out profiles
- terrain-following P-wave velocity sections along each profile, with
  interface-depth uncertainty across the ensemble
- gridded mobile-regolith thickness, weathered-bedrock thickness, and depth to
  fresh bedrock, with their ensemble spread

## Two Run Modes

**Load archived results (default).** With `RUN_SCEUA = False` the notebook reads
`outputs/dc_sceua_pc055_lb/sceua_results.csv`, rebuilds the best-fit model, and
recovers every reported quantity without repeating the search. Runtime is a few
minutes.

**Recalibrate.** Set `RUN_SCEUA = True` and choose `RUN_PRESET`. The `smoke`
preset runs 20 objective evaluations and exists to check that the full Landlab,
rock-physics, and pyGIMLi path executes. The `production` preset runs 2,500
evaluations and repeats the full search; each evaluation is one 6,000-year
landscape-evolution run plus seven travel-time simulations, so expect many
hours. Point `output_dir` at a scratch directory to keep the archived results
intact.


## Function Map

The workflow is intentionally split into small functions so each cell has one job.

- `load_sceua_config`: fills defaults from `config.yaml`.
- `load_sceua_base_inputs`: loads the runtime DEM, uses ArcGIS flow rasters to compute `Zs`, loads seismic line data, and builds reusable pyGIMLi meshes.
- `run_landlab_soil`: runs the Landlab soil-production/diffusion model for one candidate.
- `rd_geometry_from_candidate`: builds the strict RD relief-ratio geometry, `Zb = r Zs`, `D_fresh_raw = Zs - Zb`, enforces the physical layer ordering `D_fresh >= H_soil`, and computes ordering diagnostics.
- `build_model_from_candidate`: combines Landlab soil, RD geometry, interface-anchored porosity, and granite RPV rock physics into a 3-D Vp model.
- `run_sceua_calibration`: wraps the candidate evaluator for SPOTPY SCE-UA and saves candidate tables.
- `predict_lines_with_behavioral_sets`: predicts training or validation travel times using retained behavioral parameter sets.
- `propagate_behavioral_to_3d`: maps behavioral parameter uncertainty into full-domain CZ, porosity, and Vp uncertainty.

In [ ]:
from __future__ import annotations

from pathlib import Path
from copy import deepcopy
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data_io import read_config
from src.dem_tools import sample_dem_along_line
from src.sceua_landlab_rd_rpv import (
    SCEUA_PARAMETER_NAMES,
    build_model_from_candidate,
    build_line_section_from_model,
    fixed_parameter_values,
    forward_predict_line,
    get_line_split,
    load_sceua_base_inputs,
    load_sceua_config,
    parameter_bounds,
    predict_lines_with_behavioral_sets,
    propagate_behavioral_to_3d,
    run_sceua_calibration,
    save_3d_uncertainty_outputs,
    save_prediction_outputs,
    select_behavioral_sets,
    summarize_behavioral_ensemble,
    validate_forward_depth,
)

## Inputs You Can Change

Use `smoke` first to check the full workflow. Move to `production` for the paper calibration. The runtime configuration below is isolated from `config.yaml` and writes to a dedicated long-line output directory.

In [ ]:
RUN_PRESET = "production"
# Recalibration is opt-in. Existing paper results are loaded by default.
RUN_SCEUA = False
LOAD_EXISTING_IF_AVAILABLE = True
ARCHIVE_EXISTING_SCEUA_DATABASE = False

RUN_TRAINING_PREDICTION = False
RUN_VALIDATION_PREDICTION = False
RUN_3D_PROPAGATION = False
RUN_BEHAVIORAL_LINE_SECTION_EXPORT = True

MAX_BEHAVIORAL_FORWARD_SETS = None
N_3D_BEHAVIORAL_SETS = None

FIG_DPI = 180


## 1. Load Config, DEM, ArcGIS Flow Rasters, and Seismic Lines

This cell builds reusable pyGIMLi meshes for the five long training lines and two held-out validation lines. The two short smallline datasets remain excluded from discovery and from the SCE-UA objective. Runtime overrides are isolated from `config.yaml`: this run uses `Fill_cz_modeling_2.tif` for the model grid and `large_tiff/Fill_cz_modeling_large.tif`, `large_tiff/Flow_Direction_large.tif`, `large_tiff/Flow_Accumulation_Flow_large.tif`, and `large_tiff/RasterC_1200_large.tif` to compute `Z_s`, followed by light smoothing on the 5 m model grid.

In [ ]:
config = read_config(ROOT / "config.yaml")
workflow_config = config["sceua_landlab_rd_rpv"]
landlab_runtime = config["landlab_evolution"]
sceua_config = load_sceua_config(config)
validate_forward_depth(sceua_config)

required_inputs = [
    ROOT / config["dem_file"],
    *[ROOT / landlab_runtime[name] for name in (
        "flow_dem_file",
        "flow_direction_file",
        "flow_accumulation_file",
        "channel_raster_file",
    )],
]
for required_input in required_inputs:
    if not required_input.exists():
        raise FileNotFoundError(required_input)

training_lines, validation_lines = get_line_split(config)
assert len(training_lines) == 5
assert len(validation_lines) == 2
assert fixed_parameter_values(sceua_config) == {}
assert parameter_bounds(sceua_config)["Hs"] == (0.125, 1.0)
assert parameter_bounds(sceua_config)["D"] == (1.0e-4, 2.0e-3)
assert parameter_bounds(sceua_config)["r"] == (0.25, 0.95)
assert parameter_bounds(sceua_config)["phi_soil_top"] == (0.45, 0.60)
assert parameter_bounds(sceua_config)["phi_weathered_top"] == (0.25, 0.45)
assert parameter_bounds(sceua_config)["phi_fresh"] == (0.05, 0.10)

OUTPUT_DIR = ROOT / sceua_config["output_dir"]
DATA_DIR = OUTPUT_DIR / "data"
FIG_DIR = OUTPUT_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

def notebook_output_path(path):
    """Return a parent-created, Windows-long-path-safe output path."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if os.name != "nt":
        return path
    text = str(path.resolve())
    if len(text) < 240 or text.startswith("\\?\\"):
        return text
    if text.startswith("\\"):
        return "\\?\\UNC\\" + text[2:]
    return "\\?\\" + text

base_inputs = load_sceua_base_inputs(config, ROOT)
assert set(base_inputs["lines"]) == set(training_lines + validation_lines)
assert base_inputs["rpv_params"]["soil"]["critical_porosity"] == 0.55
assert base_inputs["rpv_params"]["soil"]["hertz_mindlin_bound"] == "lower"
assert base_inputs["rpv_params"]["weathered_bedrock"]["alpha_top"] == 0.015
assert base_inputs["rpv_params"]["weathered_bedrock"]["alpha_bottom"] == 0.015
assert base_inputs["rpv_params"]["fresh_bedrock"]["alpha"] == 0.015

pd.DataFrame([
    {
        "training_lines": ", ".join(training_lines),
        "validation_lines": ", ".join(validation_lines),
        "dem_file": config["dem_file"],
        "target_dem_resolution_m": config["target_dem_resolution"],
        "flow_source": landlab_runtime["flow_source"],
        "flow_dem_file": landlab_runtime["flow_dem_file"],
        "flow_direction_file": landlab_runtime["flow_direction_file"],
        "flow_accumulation_file": landlab_runtime["flow_accumulation_file"],
        "channel_raster_file": landlab_runtime["channel_raster_file"],
        "flow_channel_source": base_inputs["flow"].channel_source,
        "channel_cell_count": int(np.sum(base_inputs["flow"].channel_mask)),
        "mean_Zs_m": float(np.nanmean(base_inputs["Zs"])),
        "max_Zs_m": float(np.nanmax(base_inputs["Zs"])),
        "forward_bottom_m": sceua_config["forward_model"]["bottom_m"],
    }
])


### Active Interface-Anchored Porosity and Granite RPV Settings

Confirm the seven sampled parameters (`P0`, `Hs`, `D`, `r`, and porosity endpoints), summer saturation profile, ArcGIS flow rasters, and Holbrook/Flinchum granite rock-physics constants before recalibration. `phi_soil_top` is sampled from 0.45 to 0.60, `phi_fresh` is sampled from 0.05 to 0.10, mobile regolith uses `phi_c=0.55` with the lower Hertz-Mindlin bound, and both weathered and fresh bedrock use `alpha=0.015`. Boundary diagnostics for `r`, `phi_weathered_top`, and `phi_fresh` should be reported with the paper results.

In [ ]:
active_rpv = base_inputs["rpv_params"]
active_bounds = parameter_bounds(sceua_config)
active_fixed = fixed_parameter_values(sceua_config)
sampled_parameter_names = [
    name for name in SCEUA_PARAMETER_NAMES if name not in active_fixed
]

active_parameter_table = pd.DataFrame(
    [
        {
            "parameter": name,
            "role": "fixed" if name in active_fixed else "SCE-UA",
            "lower": active_fixed.get(name, active_bounds[name][0]),
            "upper": active_fixed.get(name, active_bounds[name][1]),
        }
        for name in SCEUA_PARAMETER_NAMES
    ]
)

fixed_rpv_table = pd.DataFrame(
    [
        {
            "basis": active_rpv.get("basis", "default"),
            "vp_max_m_per_s": active_rpv["vp_max"],
            "soil_Sw_top": active_rpv["soil"]["Sw_top"],
            "soil_Sw_bottom": active_rpv["soil"]["Sw_bottom"],
            "soil_critical_porosity": active_rpv["soil"]["critical_porosity"],
            "soil_hertz_mindlin_bound": active_rpv["soil"]["hertz_mindlin_bound"],
            "weathered_Sw_top": active_rpv["weathered_bedrock"]["Sw_top"],
            "weathered_Sw_bottom": active_rpv["weathered_bedrock"]["Sw_bottom"],
            "weathered_alpha_top": active_rpv["weathered_bedrock"]["alpha_top"],
            "weathered_alpha_bottom": active_rpv["weathered_bedrock"]["alpha_bottom"],
            "fresh_Sw": active_rpv["fresh_bedrock"]["Sw"],
            "fresh_alpha": active_rpv["fresh_bedrock"]["alpha"],
            "granite_Km_GPa": active_rpv["weathered_bedrock"]["Km"],
            "granite_Gm_GPa": active_rpv["weathered_bedrock"]["Gm"],
            "granite_rho_kg_m3": active_rpv["weathered_bedrock"]["rho"],
            "porosity_rule": "linear: surface -> soil/weathered interface -> fresh interface",
        }
    ]
)

assert active_rpv["vp_max"] == 4500.0
assert active_rpv["soil"]["critical_porosity"] == 0.55
assert active_rpv["soil"]["hertz_mindlin_bound"] == "lower"
assert active_rpv["weathered_bedrock"]["alpha_top"] == 0.015
assert active_rpv["weathered_bedrock"]["alpha_bottom"] == 0.015
assert active_rpv["fresh_bedrock"]["alpha"] == 0.015
assert sampled_parameter_names == list(SCEUA_PARAMETER_NAMES)
display(active_parameter_table)
display(fixed_rpv_table)

## 2. Run SPOTPY SCE-UA or Load Existing Results

For each candidate, the evaluator runs:

Landlab soil model -> RD geometry -> interface-anchored porosity -> granite RPV Vp -> pyGIMLi first-arrival forward model -> line-balanced seismic normalized RMSE.

In [ ]:
root_results_path = OUTPUT_DIR / "sceua_results.csv"
root_behavioral_path = OUTPUT_DIR / "behavioral_parameter_sets.csv"
root_best_path = OUTPUT_DIR / "best_fit_parameters.csv"
root_behavioral_summary_path = OUTPUT_DIR / "behavioral_parameter_summary.csv"

data_results_path = DATA_DIR / "sceua_results.csv"
data_behavioral_path = DATA_DIR / "behavioral_parameter_sets.csv"
data_best_path = DATA_DIR / "best_fit_parameters.csv"
data_behavioral_summary_path = DATA_DIR / "behavioral_parameter_summary.csv"


def prefer_existing_path(primary: Path, fallback: Path) -> Path:
    return primary if primary.exists() else fallback


results_path = prefer_existing_path(root_results_path, data_results_path)
behavioral_path = prefer_existing_path(root_behavioral_path, data_behavioral_path)
best_path = prefer_existing_path(root_best_path, data_best_path)

if RUN_SCEUA and ARCHIVE_EXISTING_SCEUA_DATABASE: 
    spotpy_db_path = OUTPUT_DIR / "spotpy_sceua.csv"
    if spotpy_db_path.exists():
        archive_dir = OUTPUT_DIR / "logs"
        archive_dir.mkdir(parents=True, exist_ok=True)
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        archive_path = archive_dir / f"spotpy_sceua_previous_{timestamp}.csv"
        spotpy_db_path.replace(archive_path)
        print(f"Archived existing SCE-UA database to {archive_path}")

if RUN_SCEUA:
    sceua_results = run_sceua_calibration(
        training_lines,
        base_inputs,
        config,
        preset=RUN_PRESET,
        output_dir=OUTPUT_DIR,
        save_outputs=True,
    )
    results_df = sceua_results["records"]
    behavioral_sets = sceua_results["behavioral_sets"]
    best_parameters = sceua_results["best_parameters"]
    results_df.to_csv(notebook_output_path(data_results_path), index=False)
    behavioral_sets.to_csv(notebook_output_path(data_behavioral_path), index=False)
    best_parameters.to_csv(notebook_output_path(data_best_path), index=False)
    if "behavioral_summary" in sceua_results and sceua_results["behavioral_summary"] is not None:
        sceua_results["behavioral_summary"].to_csv(notebook_output_path(data_behavioral_summary_path), index=False)
elif LOAD_EXISTING_IF_AVAILABLE and results_path.exists() and behavioral_path.exists():
    results_df = pd.read_csv(results_path)
    behavioral_sets = pd.read_csv(behavioral_path)
    best_parameters = pd.read_csv(best_path) if best_path.exists() else results_df.iloc[[0]].copy()
else:
    raise FileNotFoundError(
        "Run SCE-UA first or place existing CSV files in either the output root or output data folder."
    )

behavioral_summary = summarize_behavioral_ensemble(behavioral_sets)
paper_parameter_summary = active_parameter_table.merge(
    behavioral_summary, on="parameter", how="left"
)
paper_parameter_summary["best_fit"] = [
    float(best_parameters.iloc[0][name])
    for name in paper_parameter_summary["parameter"]
]
paper_parameter_summary["near_bound_5pct"] = [
    True
    if row.role == "fixed"
    else min(row.best_fit - row.lower, row.upper - row.best_fit)
    <= 0.05 * (row.upper - row.lower)
    for row in paper_parameter_summary.itertuples()
]
paper_parameter_summary.to_csv(
    notebook_output_path(DATA_DIR / "paper_parameter_summary.csv"), index=False
)
display(best_parameters)
display(paper_parameter_summary)


## 3. Calibration Diagnostics

These plots show how the seismic objective varies across the seven sampled parameters and how the retained behavioral ensemble is distributed. `phi_fresh` is now sampled from 0.05 to 0.10 rather than fixed.

In [ ]:
parameter_names = list(sampled_parameter_names)
ncols = 3
nrows = int(np.ceil(len(parameter_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.25 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, name in zip(axes, parameter_names):
    ax.scatter(results_df[name], results_df["objective_value"], s=12, alpha=0.45, color="tab:blue")
    if name in behavioral_sets:
        ax.scatter(behavioral_sets[name], behavioral_sets["objective_value"], s=18, alpha=0.75, color="tab:orange")
    ax.set_xlabel(name)
    ax.set_ylabel("objective")
    ax.grid(alpha=0.25)
for ax in axes[len(parameter_names):]:
    ax.axis("off")
fig.savefig(notebook_output_path(FIG_DIR / "sceua_parameter_objective_scatter.png"), dpi=FIG_DPI)
plt.show()

line_metric_cols = [c for c in results_df.columns if c.endswith("_normalized_rmse") and c != "mean_training_normalized_rmse"]
if line_metric_cols:
    display(best_parameters[["objective_value", "mean_training_rmse_s", *line_metric_cols]].T)

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.25 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, name in zip(axes, parameter_names):
    ax.hist(results_df[name].dropna(), bins=24, color="0.80", label="all candidates")
    ax.hist(behavioral_sets[name].dropna(), bins=16, color="tab:orange", alpha=0.75, label="behavioral")
    ax.set_xlabel(name)
    ax.set_ylabel("count")
    ax.grid(alpha=0.20)
for ax in axes[len(parameter_names):]:
    ax.axis("off")
axes[0].legend(frameon=False)
fig.savefig(notebook_output_path(FIG_DIR / "sceua_behavioral_parameter_distributions.png"), dpi=FIG_DPI)
plt.show()

corr = behavioral_sets[parameter_names].corr()
fig, ax = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(np.arange(len(parameter_names)), parameter_names, rotation=45, ha="right")
ax.set_yticks(np.arange(len(parameter_names)), parameter_names)
fig.colorbar(im, ax=ax, label="correlation")
fig.savefig(notebook_output_path(FIG_DIR / "sceua_behavioral_parameter_correlation.png"), dpi=FIG_DPI)
plt.show()

if len(behavioral_sets) > 1:
    axes = pd.plotting.scatter_matrix(
        behavioral_sets[parameter_names],
        figsize=(9, 9),
        diagonal="hist",
        alpha=0.65,
        s=18,
    )
    for ax in np.ravel(axes):
        ax.grid(alpha=0.15)
    plt.gcf().savefig(notebook_output_path(FIG_DIR / "sceua_behavioral_parameter_pairs.png"), dpi=FIG_DPI)
    plt.show()

## 4. Predict Training Lines With the Behavioral Ensemble

This step reruns the forward model for the retained behavioral parameter sets. By default `MAX_BEHAVIORAL_FORWARD_SETS = None`, so all retained behavioral sets are used; set it to an integer only for quick smoke checks.

In [ ]:
if RUN_TRAINING_PREDICTION:
    training_prediction = predict_lines_with_behavioral_sets(
        behavioral_sets,
        training_lines,
        base_inputs,
        config,
        max_sets=MAX_BEHAVIORAL_FORWARD_SETS,
    )
    training_prediction_summary = save_prediction_outputs(
        training_prediction,
        OUTPUT_DIR,
        prefix="training_behavioral",
    )
    display(training_prediction_summary)
else:
    training_prediction = None
    training_prediction_summary = None

## 5. Predict Held-Out Validation Lines Without Refitting

The validation lines are not used in the SCE-UA objective. This cell only propagates the calibrated behavioral ensemble to those held-out lines.

In [ ]:
if RUN_VALIDATION_PREDICTION:
    validation_prediction = predict_lines_with_behavioral_sets(
        behavioral_sets,
        validation_lines,
        base_inputs,
        config,
        max_sets=MAX_BEHAVIORAL_FORWARD_SETS,
    )
    validation_prediction_summary = save_prediction_outputs(
        validation_prediction,
        OUTPUT_DIR,
        prefix="validation_behavioral",
    )
    display(validation_prediction_summary)
else:
    validation_prediction = None
    validation_prediction_summary = None

paper_line_fit_frames = []
if training_prediction_summary is not None:
    paper_line_fit_frames.append(
        training_prediction_summary.assign(dataset="calibration")
    )
if validation_prediction_summary is not None:
    paper_line_fit_frames.append(
        validation_prediction_summary.assign(dataset="validation")
    )
if paper_line_fit_frames:
    paper_line_fit_summary = pd.concat(
        paper_line_fit_frames, ignore_index=True
    )
    paper_line_fit_summary.to_csv(
        notebook_output_path(DATA_DIR / "paper_line_fit_summary.csv"), index=False
    )
    display(paper_line_fit_summary)

In [ ]:
def plot_prediction_envelope(prediction, title_prefix, output_name):
    if prediction is None:
        return
    n_lines = len(prediction["lines"])
    fig, axes = plt.subplots(n_lines, 1, figsize=(9, max(2.4 * n_lines, 3.0)), constrained_layout=True)
    if n_lines == 1:
        axes = [axes]
    for ax, (line_id, line_result) in zip(axes, prediction["lines"].items()):
        offset = line_result["offset"]
        observed = line_result["observed"]
        order = np.argsort(offset)
        ax.fill_between(
            offset[order],
            line_result["p05"][order],
            line_result["p95"][order],
            color="tab:blue",
            alpha=0.20,
        )
        ax.scatter(
            offset, line_result["p50"], s=10, color="tab:blue",
            alpha=0.75, label="behavioral median"
        )
        ax.scatter(offset, observed, s=12, color="black", label="observed")
        ax.set_title(f"{title_prefix}: {line_id}")
        ax.set_xlabel("offset (m)")
        ax.set_ylabel("travel time (s)")
        ax.grid(alpha=0.25)
    axes[0].legend(frameon=False)
    fig.savefig(notebook_output_path(FIG_DIR / output_name), dpi=FIG_DPI)
    plt.show()

plot_prediction_envelope(training_prediction, "Training prediction", "training_behavioral_prediction_envelopes.png")
plot_prediction_envelope(validation_prediction, "Validation prediction", "validation_behavioral_prediction_envelopes.png")

## 6. Propagate Behavioral Sets to 3-D CZ, Porosity, and Vp Uncertainty

This step computes full-domain maps of mean and spread in soil thickness, weathered-bedrock thickness, fresh-bedrock depth, porosity, and Vp slices.

In [ ]:
if RUN_3D_PROPAGATION:
    uncertainty_3d = propagate_behavioral_to_3d(
        behavioral_sets,
        base_inputs,
        config,
        n_sets=N_3D_BEHAVIORAL_SETS,
    )
    save_3d_uncertainty_outputs(uncertainty_3d, OUTPUT_DIR)
    display(uncertainty_3d["diagnostics"])
else:
    uncertainty_3d = None

In [ ]:
if uncertainty_3d is not None:
    x = uncertainty_3d["x"]
    y = uncertainty_3d["y"]
    extent = [float(np.nanmin(x)), float(np.nanmax(x)), float(np.nanmin(y)), float(np.nanmax(y))]
    fields = [
        ("H_soil_mean", "mean soil thickness (m)", "viridis"),
        ("H_weathered_mean", "mean weathered bedrock (m)", "viridis"),
        ("D_fresh_p50", "median depth to fresh bedrock (m)", "viridis"),
        ("D_fresh_std", "std depth to fresh bedrock (m)", "magma"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
    for ax, (key, title, cmap) in zip(axes.ravel(), fields):
        image = ax.imshow(uncertainty_3d[key], origin="lower", extent=extent, cmap=cmap)
        ax.set_title(title)
        ax.set_xlabel("Easting (m)")
        ax.set_ylabel("Northing (m)")
        fig.colorbar(image, ax=ax, shrink=0.82)
    fig.savefig(notebook_output_path(FIG_DIR / "behavioral_3d_cz_uncertainty_maps.png"), dpi=FIG_DPI)
    plt.show()

In [ ]:
if uncertainty_3d is not None:
    depth = uncertainty_3d["depth"]
    x = uncertainty_3d["x"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    im0 = axes[0].imshow(
        uncertainty_3d["Vp_section_p50"],
        origin="upper",
        aspect="auto",
        extent=[float(np.nanmin(x)), float(np.nanmax(x)), float(depth[-1]), float(depth[0])],
        cmap="plasma",
    )
    axes[0].set_title("median Vp section")
    axes[0].set_xlabel("Easting-like section coordinate (m)")
    axes[0].set_ylabel("depth below surface (m)")
    fig.colorbar(im0, ax=axes[0], label="Vp (m/s)")

    im1 = axes[1].imshow(
        uncertainty_3d["Vp_section_std"],
        origin="upper",
        aspect="auto",
        extent=[float(np.nanmin(x)), float(np.nanmax(x)), float(depth[-1]), float(depth[0])],
        cmap="magma",
    )
    axes[1].set_title("Vp standard deviation section")
    axes[1].set_xlabel("Easting-like section coordinate (m)")
    axes[1].set_ylabel("depth below surface (m)")
    fig.colorbar(im1, ax=axes[1], label="Vp std (m/s)")
    fig.savefig(notebook_output_path(FIG_DIR / "behavioral_vp_section_uncertainty.png"), dpi=FIG_DPI)
    plt.show()

In [ ]:
if uncertainty_3d is not None:
    required_profile_keys = [
        "Phi_profile_p05",
        "Phi_profile_p50",
        "Phi_profile_p95",
        "Vp_profile_p05",
        "Vp_profile_p50",
        "Vp_profile_p95",
    ]
    missing_profile_keys = [key for key in required_profile_keys if key not in uncertainty_3d]
    if missing_profile_keys:
        import importlib
        import src.sceua_landlab_rd_rpv as sceua_workflow

        print(
            "uncertainty_3d is missing the new porosity/Vp profile fields; "
            "reloading workflow helpers and recomputing 3-D propagation."
        )
        sceua_workflow = importlib.reload(sceua_workflow)
        propagate_behavioral_to_3d = sceua_workflow.propagate_behavioral_to_3d
        save_3d_uncertainty_outputs = sceua_workflow.save_3d_uncertainty_outputs
        uncertainty_3d = propagate_behavioral_to_3d(
            behavioral_sets,
            base_inputs,
            config,
            n_sets=N_3D_BEHAVIORAL_SETS,
        )
        save_3d_uncertainty_outputs(uncertainty_3d, OUTPUT_DIR)

    missing_profile_keys = [key for key in required_profile_keys if key not in uncertainty_3d]
    if missing_profile_keys:
        raise KeyError(
            "3-D uncertainty output still lacks profile fields: "
            + ", ".join(missing_profile_keys)
            + ". Restart the notebook kernel and run from the import cell."
        )

    depth = uncertainty_3d["depth"]
    fig, axes = plt.subplots(1, 2, figsize=(9, 5), constrained_layout=True)

    axes[0].fill_betweenx(
        depth,
        uncertainty_3d["Phi_profile_p05"],
        uncertainty_3d["Phi_profile_p95"],
        color="tab:green",
        alpha=0.22,
    )
    axes[0].plot(uncertainty_3d["Phi_profile_p50"], depth, color="tab:green", lw=1.8)
    axes[0].invert_yaxis()
    axes[0].set_xlabel("porosity")
    axes[0].set_ylabel("depth below surface (m)")
    axes[0].set_title("behavioral porosity-depth envelope")
    axes[0].grid(alpha=0.25)

    axes[1].fill_betweenx(
        depth,
        uncertainty_3d["Vp_profile_p05"],
        uncertainty_3d["Vp_profile_p95"],
        color="tab:purple",
        alpha=0.22,
    )
    axes[1].plot(uncertainty_3d["Vp_profile_p50"], depth, color="tab:purple", lw=1.8)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("Vp (m/s)")
    axes[1].set_ylabel("depth below surface (m)")
    axes[1].set_title("behavioral Vp-depth envelope")
    axes[1].grid(alpha=0.25)

    fig.savefig(notebook_output_path(FIG_DIR / "behavioral_porosity_vp_depth_envelopes.png"), dpi=FIG_DPI)
    plt.show()


## Output Files

Main CSV outputs are written under `outputs/dc_sceua_pc055_lb/`:

- `sceua_results.csv`
- `behavioral_parameter_sets.csv`
- `best_fit_parameters.csv`
- `behavioral_parameter_summary.csv`
- `data/paper_parameter_summary.csv`
- `data/paper_line_fit_summary.csv`

Prediction and 3-D uncertainty arrays are written under the `data/` subfolder, and figures are written under `figures/`. This output directory is separate from the previous Landlab-routed and ArcGIS-channel-only runs.

## Export Behavioral Median Line Velocity Sections

This cell uses the already retained behavioral parameter sets to write per-line median Vp sections for paper figures. It rebuilds the 3-D CZ/RPV models and extracts line sections only; it does not rerun SCE-UA and does not run travel-time forward modeling.

In [ ]:
BEHAVIORAL_LINE_SECTION_PATH = DATA_DIR / "behavioral_line_sections.npz"


def _safe_line_key(line_id: str) -> str:
    return "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in line_id)


def _line_surface_from_dem(line_id: str) -> np.ndarray:
    line_xy = np.asarray(base_inputs["lines"][line_id]["line_xy"], dtype=float)
    surface = sample_dem_along_line(line_xy[:, 0], line_xy[:, 1], base_inputs["dem"])
    if not np.all(np.isfinite(surface)):
        raise ValueError(f"DEM sampling failed for {line_id}.")
    return surface


def _line_section_for_model(model: dict, line_id: str) -> dict[str, np.ndarray]:
    section = build_line_section_from_model(model, base_inputs["lines"][line_id], base_inputs)
    out = dict(section)
    out["surface_elevation"] = _line_surface_from_dem(line_id)
    return out


def export_behavioral_line_sections(
    behavioral_frame: pd.DataFrame,
    output_path: Path,
    *,
    max_sets: int | None = None,
) -> dict[str, np.ndarray]:
    """Save behavioral Vp, porosity, and interface summaries for Notebook 14."""
    frame = behavioral_frame.copy()
    if max_sets is not None:
        frame = frame.head(max_sets).copy()
    if frame.empty:
        raise ValueError("No behavioral parameter sets are available for export.")

    fields = ["vp", "phi", "H_soil", "H_weathered", "D_fresh"]
    stacks = {
        line_id: {field: [] for field in fields}
        for line_id in training_lines + validation_lines
    }
    reference_sections: dict[str, dict[str, np.ndarray]] = {}

    for sample_number, (_, row) in enumerate(frame.iterrows(), start=1):
        model = build_model_from_candidate(row, base_inputs, config)
        for line_id in training_lines + validation_lines:
            section = _line_section_for_model(model, line_id)
            reference_sections.setdefault(line_id, section)
            for field in fields:
                stacks[line_id][field].append(np.asarray(section[field], dtype=float))
        if sample_number == 1 or sample_number % 10 == 0 or sample_number == len(frame):
            print(f"exported line sections for behavioral set {sample_number}/{len(frame)}")

    arrays: dict[str, np.ndarray] = {
        "line_ids": np.asarray(training_lines + validation_lines, dtype=str),
        "n_behavioral_sets": np.asarray([len(frame)], dtype=int),
    }
    for line_id in training_lines + validation_lines:
        key = _safe_line_key(line_id)
        reference = reference_sections[line_id]
        arrays[f"{key}__distance"] = np.asarray(reference["distance"], dtype=float)
        arrays[f"{key}__depth"] = np.asarray(reference["depth"], dtype=float)
        arrays[f"{key}__surface_elevation"] = np.asarray(reference["surface_elevation"], dtype=float)
        for field in fields:
            stack = np.stack(stacks[line_id][field], axis=0)
            arrays[f"{key}__{field}_p05"] = np.nanpercentile(stack, 5, axis=0)
            arrays[f"{key}__{field}_p50"] = np.nanpercentile(stack, 50, axis=0)
            arrays[f"{key}__{field}_p95"] = np.nanpercentile(stack, 95, axis=0)
            arrays[f"{key}__{field}_mean"] = np.nanmean(stack, axis=0)
            arrays[f"{key}__{field}_std"] = np.nanstd(stack, axis=0)

    np.savez_compressed(notebook_output_path(output_path), **arrays)
    return arrays


if RUN_BEHAVIORAL_LINE_SECTION_EXPORT:
    behavioral_line_sections = export_behavioral_line_sections(
        behavioral_sets,
        BEHAVIORAL_LINE_SECTION_PATH,
        max_sets=N_3D_BEHAVIORAL_SETS,
    )
    print(f"Saved behavioral line sections to {BEHAVIORAL_LINE_SECTION_PATH.relative_to(ROOT)}")
    print(f"n_behavioral_sets={int(behavioral_line_sections['n_behavioral_sets'][0])}")
else:
    print("Behavioral line-section export is disabled.")
